In [12]:
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", 
    "chromadb", "langchain", "langchain-community", 
    "sentence-transformers", "pymupdf", "python-dotenv", "openai"])

CompletedProcess(args=['F:\\d files\\my projects\\smart document insights\\venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'chromadb', 'langchain', 'langchain-community', 'sentence-transformers', 'pymupdf', 'python-dotenv', 'openai'], returncode=0)

In [13]:
# Test that everything is installed correctly
import langchain
import chromadb
import sentence_transformers
import fitz  # this is pymupdf
import openai
from dotenv import load_dotenv
import os

# Load your API key from .env file
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("✅ API key loaded successfully!")
else:
    print("❌ API key NOT found — check your .env file")

print("✅ All packages imported successfully!")

✅ API key loaded successfully!
✅ All packages imported successfully!


In [14]:
##This will show us exactly where Jupyter is looking and what files it can see. Paste the output here! 
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir())

Current folder: F:\d files\my projects\smart document insights
Files here: ['.env', '.ipynb_checkpoints', '01_ingestion.ipynb', '02_webapp.ipynb', 'app.py', 'chroma_db', 'index.html', 'launcher.bat', 'sample.pdf.pdf', 'smart-doc-insights.html', 'uploads', 'venv', '__pycache__']


In [15]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("✅ API key loaded successfully!")
else:
    print("❌ Still not found")

✅ API key loaded successfully!


In [16]:
## this is our PDF reader, the first real piece of the project:
import fitz  # pymupdf

def read_pdf(file_path):
    doc = fitz.open(file_path)
    text = ""
    for page_num, page in enumerate(doc):
        text += f"\n--- Page {page_num + 1} ---\n"
        text += page.get_text()
    print(f"✅ PDF loaded! Total pages: {len(doc)}")
    print(f"✅ Total characters extracted: {len(text)}")
    print("\n--- PREVIEW (first 500 characters) ---")
    print(text[:500])
    return text

# TEST IT — put any PDF in your project folder and change the name below
text = read_pdf("sample.pdf.pdf")

✅ PDF loaded! Total pages: 66
✅ Total characters extracted: 174949

--- PREVIEW (first 500 characters) ---

--- Page 1 ---
1 
 
Introduction: 
Registration refers to the recording of the contents of a document with a Registering 
Officer appointed by the Government. The main purpose of registration is to ensure information 
about all deals are recorded and maintained apart from giving the document its authenticity. It 
gives information to the people regarding legal rights and obligations arising or affecting a 
particular property. The registered documents may afterwards be of legal importance, and 


In [17]:
### to know the file location and it's name
import os
print(os.listdir())

['.env', '.ipynb_checkpoints', '01_ingestion.ipynb', '02_webapp.ipynb', 'app.py', 'chroma_db', 'index.html', 'launcher.bat', 'sample.pdf.pdf', 'smart-doc-insights.html', 'uploads', 'venv', '__pycache__']


In [18]:
### this splits that giant text into small chunks that the AI can work with

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_into_chunks(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,        # each chunk = ~500 characters
        chunk_overlap=50,      # chunks overlap by 50 chars so nothing is lost
        length_function=len
    )
    chunks = splitter.split_text(text)
    print(f"✅ Total chunks created: {len(chunks)}")
    print(f"\n--- PREVIEW of chunk #1 ---")
    print(chunks[0])
    print(f"\n--- PREVIEW of chunk #2 ---")
    print(chunks[1])
    return chunks

chunks = split_into_chunks(text)

✅ Total chunks created: 415

--- PREVIEW of chunk #1 ---
--- Page 1 ---
1 
 
Introduction: 
Registration refers to the recording of the contents of a document with a Registering 
Officer appointed by the Government. The main purpose of registration is to ensure information 
about all deals are recorded and maintained apart from giving the document its authenticity. It 
gives information to the people regarding legal rights and obligations arising or affecting a

--- PREVIEW of chunk #2 ---
particular property. The registered documents may afterwards be of legal importance, and also 
aid in preventing fraud. The process of registering a document is done under the provisions of 
the Registration Act, 1908. The main objects of the law of registration are – 
(a) to provide a conclusive proof of genuineness of documents; 
(b) to afford publicity of transaction in respect of properties; 
(c) to prevent fraud;


In [19]:
###converting those chunks into embeddings and storing them in ChromaDB. Paste this in the next cell:

from sentence_transformers import SentenceTransformer
import chromadb

# Load the free local embedding model
print("Loading embedding model... (first time takes 1-2 mins)")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded!")

# Create ChromaDB database
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="documents")
print("✅ ChromaDB ready!")

# Embed all chunks and store them
print("Embedding and storing chunks... please wait...")
for i, chunk in enumerate(chunks):
    embedding = embedder.encode(chunk).tolist()
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[f"chunk_{i}"],
        metadatas=[{"source": "sample.pdf", "chunk_index": i}]
    )

print(f"✅ All {len(chunks)} chunks stored in ChromaDB!")

Task was destroyed but it is pending!
task: <Task pending name='Task-125' coro=<_async_in_context.<locals>.run_in_context() done, defined at F:\d files\my projects\smart document insights\venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-126' coro=<Kernel.shell_main() running at F:\d files\my projects\smart document insights\venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at F:\d files\my projects\smart document insights\venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
F:\d files\my projects\smart document insights\venv\Lib\site-packages\httpx\_urls.py:77: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  def __init__(self, url: URL | str = "", **kwargs: typing.Any) -> None:
Task was destroyed but it is pending!
task: <Task pending name='Task-126' coro=<Kernel.shell_main() running at F:\d files\my projects\smart document insights\venv\Lib\site-packages\ipy

Loading embedding model... (first time takes 1-2 mins)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded!
✅ ChromaDB ready!
Embedding and storing chunks... please wait...
✅ All 415 chunks stored in ChromaDB!


In [20]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "ollama"])

CompletedProcess(args=['F:\\d files\\my projects\\smart document insights\\venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'ollama'], returncode=0)

In [21]:
## ###asking a question and getting an AI answer from your document. Paste this in the next cell:

import ollama

def ask_document(question):
    print(f"🔍 Question: {question}\n")
    
    # Step 1: Embed the question
    question_embedding = embedder.encode(question).tolist()
    
    # Step 2: Find most relevant chunks from ChromaDB
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=5
    )
    context = "\n\n".join(results['documents'][0])
    
    # Step 3: Send to Ollama (free, local!)
    response = ollama.chat(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": "Answer using ONLY the context provided. If the answer isn't in the context, say 'I don't know'. Be concise."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    
    answer = response['message']['content']
    print(f"🤖 Answer:\n{answer}")
    print(f"\n📄 Based on {len(results['documents'][0])} chunks from the document")
    return answer

# ASK YOUR FIRST QUESTION!
ask_document("What are the objects of registration law?")

🔍 Question: What are the objects of registration law?

🤖 Answer:
To provide a conclusive proof of genuineness of documents; to afford publicity of transaction in respect of properties; and to prevent fraud.

📄 Based on 5 chunks from the document


'To provide a conclusive proof of genuineness of documents; to afford publicity of transaction in respect of properties; and to prevent fraud.'

In [22]:
import chromadb
client = chromadb.PersistentClient(path="./chroma_db")
client.delete_collection("documents")
client.get_or_create_collection("documents")
print("✅ Database cleared!")

✅ Database cleared!


In [23]:
import fitz
doc = fitz.open("uploads/certificate-zowzbn7gr8qu-1772615611.pdf")
for page_num, page in enumerate(doc):
    print(f"--- Page {page_num+1} ---")
    print(repr(page.get_text()))

--- Page 1 ---
'siddarth sharma\nIssued: March 4, 2026\nCertificate No: zowzbn7gr8qu\nView: https://verify.skilljar.com/c/zowzbn7gr8qu\n'
